# Dyson Protocol Storage Module Guide

This notebook demonstrates the key features of the Dyson Protocol's Storage module, which provides on-chain persistent key-value storage functionality. The Storage module allows dApps and scripts to maintain state between transactions and serves as a foundation for building robust decentralized applications.

## Introduction to Storage

The Storage module provides a simple yet powerful key-value store with these key features:

- **Key-Value Store**: Persistent on-chain storage of data
- **Owner-Based Access Control**: Only owners can modify their storage entries
- **Prefix-Based Queries**: List entries with common prefixes for efficient data organization
- **JSON Support**: Easy storage of complex data structures as JSON
- **Script Integration**: Seamless interaction with on-chain Python scripts

## Command Overview

Let's explore the available commands for the Storage module:

In [ ]:
# Transaction commands
! dysond tx storage -h

In [ ]:
# Query commands
! dysond query storage -h

## Setting Up Your Account

Let's set up our account for the following examples:

In [ ]:
# Get the address from bob account
[ADDRESS] = ! dysond keys show -a bob
print(f"Using address: {ADDRESS}")

## Basic Storage Operations

The Storage module provides basic CRUD operations. Let's explore these operations with practical examples.

### Creating/Updating Storage Entries

Let's store some user profile data. The Storage module accepts JSON data, making it ideal for structured information:

In [ ]:
# Store a user profile as JSON
import json
import shlex

PROFILE_DATA = shlex.quote(json.dumps({"name": "Bob", "bio": "Blockchain enthusiast", "skills": ["Smart Contracts", "DeFi", "Web3"]}))

out = ! dysond tx storage set \
    --index "profile/info" \
    --data $PROFILE_DATA \
    --from $ADDRESS \
    -y | dysond q wait-tx -o json

out = '\n'.join(out)
print(out)
tx_result = json.loads(out)

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break




Let's store some additional entries to demonstrate the prefix-based querying feature:

In [ ]:
# Store some example settings
SETTINGS_DATA = shlex.quote(json.dumps({"theme": "dark", "notifications": True}))

out = ! dysond tx storage set \
    --index "settings/app" \
    --data $SETTINGS_DATA \
    --from $ADDRESS \
    -y | dysond q wait-tx -o json 
out = '\n'.join(out)
print(out)
tx_result = json.loads(out)

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break

In [ ]:
# Store a blog post
POST_DATA = shlex.quote(json.dumps({
    "title": "Introduction to Blockchain",
    "content": "Blockchain is a distributed ledger technology...",
    "tags": ["blockchain", "crypto", "beginner"],
    "published_at": "2023-10-25T15:30:00Z"
}))

out = ! dysond tx storage set \
    --index "content/posts/1" \
    --data $POST_DATA \
    --from $ADDRESS \
    -y | dysond q wait-tx -o json
out = ''.join(out)
tx_result = json.loads(out)

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break



### Reading Storage Entries

Let's retrieve the storage entries we've created:

In [ ]:
# Query a specific entry
out = ! dysond query storage get $ADDRESS --index "profile/info" -o json 
print(out)
out = ''.join(out)
query_result = json.loads(out)

profile_data = json.loads(query_result['entry']['data'])
print(profile_data)

assert profile_data['name'] == "Bob", "Expected 'Bob' in output, got: " + profile_data['name']
assert profile_data['bio'] == "Blockchain enthusiast", "Expected 'Blockchain enthusiast' in output, got: " + profile_data['bio']
assert profile_data['skills'] == ["Smart Contracts", "DeFi", "Web3"], "Expected ['Smart Contracts', 'DeFi', 'Web3'] in output, got: " + str(profile_data['skills'])

### Prefix-Based Queries

The Storage module allows you to query entries with a common prefix, which is useful for organizing related data:

In [ ]:
# List all entries with the "profile/" prefix
! dysond query storage list $ADDRESS --index-prefix "profile/" -o json 

In [ ]:
# List all entries with the "content/" prefix
! dysond query storage list $ADDRESS --index-prefix "content/" -o json

In [ ]:
# List all entries (empty prefix matches everything)
! dysond query storage list $ADDRESS --index-prefix "" -o json --limit 3 --count-total

### Deleting Storage Entries

Now let's delete some of our storage entries:

In [ ]:
# Delete a single entry
out = ! dysond tx storage delete \
  --indexes "settings/app" \
  --from $ADDRESS \
  -y |  dysond q wait-tx -o json 
out = ''.join(out)
tx_result = json.loads(out)

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break

The Storage module also supports deleting multiple entries at once:

In [ ]:
# Delete multiple entries at once
out = ! dysond tx storage delete \
  --indexes "profile/info,content/posts/1" \
  --from $ADDRESS \
  -y |  dysond q wait-tx -o json 
out = ''.join(out)
tx_result = json.loads(out)

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break


Let's verify that the entries were deleted:

In [ ]:
# List all entries (should be empty now)
! dysond query storage list $ADDRESS --index-prefix "" -o json --limit 3 --count-total

## Binary Data Storage Example

While the Storage module primarily works with text data, you can store binary data by encoding it as base64 or hex. Here's a practical example using base64 encoding:

In [ ]:
import base64

# Create some binary data (a simple PNG image in this case)
binary_data = b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x10\x00\x00\x00\x10\x08\x06\x00\x00\x00\x1f\xf3\xffa'

# Encode as base64
base64_data = base64.b64encode(binary_data).decode('utf-8')
base64_json = shlex.quote(json.dumps({"data": base64_data, "mime_type": "image/png"}))

# Store the base64 encoded data
out = ! dysond tx storage set \
    --index "files/logo.png" \
    --data {base64_json} \
    --from $ADDRESS \
    -y  | dysond q wait-tx -o json 
out = ''.join(out)
tx_result = json.loads(out)

assert tx_result['code'] == 0, f"Error setting storage entry: {tx_result['raw_log']}"

for event in tx_result['events']:
    if "dysonprotocol" in event['type']:
        print(json.dumps(event, indent=2))
        break


In [ ]:
# Retrieve and decode the binary data
binary_entry = ! dysond query storage get $ADDRESS --index "files/logo.png" -o json
binary_entry_json = json.loads(''.join(binary_entry))
stored_data = json.loads(binary_entry_json['entry']['data'])
decoded_data = base64.b64decode(stored_data['data'])

print(f"Retrieved binary data of {len(decoded_data)} bytes with MIME type: {stored_data['mime_type']}")
print(f"First 20 bytes: {decoded_data[:20]}")

## Cleanup

Let's clean up by deleting all the entries we've created:

In [ ]:
# Get all entries to prepare for cleanup
entries = ! dysond query storage list $ADDRESS --index-prefix "" -o json
entries_json = json.loads(''.join(entries))

# Extract all indexes
indexes = [entry['index'] for entry in entries_json.get('entries', [])]

# Print what we're about to delete
print(f"Found {len(indexes)} entries to delete:")
for index in indexes:
    print(f"- {index}")

In [ ]:
# Delete all entries in batches of 5 to avoid command line length limits

for i in range(0, len(indexes), 5):
    index_list = indexes[i:i+5]
    print(f"Deleting [{len(index_list)}] files: {','.join(index_list)}: ", end="")

    out = ! dysond tx storage delete --indexes "{','.join(index_list)}" --from bob -y | dysond q wait-tx -o json 
    out = ''.join(out)
    tx_result = json.loads(out)

    assert tx_result['code'] == 0, f"Error deleting storage entries: {tx_result['raw_log']}"
    print(f"success")
   


In [ ]:
# Verify all entries were deleted
final_check = ! dysond query storage list $ADDRESS --index-prefix "" -o json
final_json = json.loads(''.join(final_check))
remaining = final_json.get('entries', [])

assert len(remaining) == 0, "Expected 0 entries remaining, got: " + str(remaining)
print(f"Cleanup complete. {len(remaining)} entries remaining.")

## Building a Simple dApp

Let's bring everything together by building a simple counter dApp that demonstrates the integration of dyslang with the Storage module:


In [ ]:
import json
import shlex

# Create the counter dApp script
counter_app_script = '''
from dys import _query, _msg, get_script_address
import json
from datetime import datetime

def get_counter():
    """Get the current counter value or initialize it"""
    script_address = get_script_address()
    # Try to query the existing counter
    try:
        response = _query({
            "@type": "/dysonprotocol.storage.v1.QueryStorageGetRequest",
            "owner": script_address,
            "index": "counter"
        })
        counter_data = json.loads(response["entry"]["data"])
        return counter_data["value"]
    except Exception as e:
        if "NotFound" in str(e):
            _msg({
                "@type": "/dysonprotocol.storage.v1.MsgStorageSet",
                "owner": script_address,
                "index": "counter",
                "data": json.dumps({
                    "value": 0, 
                    "updated_at": datetime.now().isoformat()
                })
            })
            return 0
        else:
            raise e



def increment_counter():
    """Increment the counter and store the new value"""
    # Get the current counter value
    current_value = get_counter()
    
    # Increment it
    new_value = current_value + 1
    
    # Store the new value
    script_address = get_script_address()

    _msg({
        "@type": "/dysonprotocol.storage.v1.MsgStorageSet",
        "owner": script_address,
         "index": "counter",
         "data": json.dumps({
            "value": new_value,
            "updated_at": datetime.now().isoformat()
        })
    })

    
    return {
        "previous_value": current_value,
        "new_value": new_value
    }

def reset_counter():
    """Reset the counter to zero"""
    script_address = get_script_address()
    _msg({
        "@type": "/dysonprotocol.storage.v1.MsgStorageDelete",
        "owner": script_address,
        "indexes": ["counter"]
    })
    return {
        "result": "Counter reset to 0"
    }
'''

# Save the counter app script
with open('/tmp/counter_app.py', 'w') as f:
    f.write(counter_app_script)
    f.close()

print(f"Setting up counter dApp for alice...")

# First, upload the script to Alice's account
out = ! dysond tx script update \
    --code-path /tmp/counter_app.py \
    --from alice \
    --gas "20000000" \
    --output json -y | dysond query wait-tx --output json
out = '\n'.join(out)
result = json.loads(out)
assert result['code'] == 0, f"Error: {result['raw_log']}"

for i in range(3):
    # Now let's test the counter app
    # 1. Get initial counter (should be 0, then set to 1)
    out = ! dysond tx script exec \
        --script-address $ALICE_ADDRESS \
        --function-name increment_counter \
        --from alice \
        --gas "10000000" \
        --output json -y | dysond query wait-tx --output json | python ../scripts/parse_exec_script_tx.py
    out = '\n'.join(out)
    result = json.loads(out)
    assert result['code'] == 0, f"Error: {result['raw_log']}"
    counter_result = result['script_result']['result']['result']
    print(f"Counter state: {counter_result['previous_value']} -> {counter_result['new_value']}")

# 4. Reset the counter for cleanup
print(f"Cleaning up...")
out = ! dysond tx script exec \
    --script-address $ALICE_ADDRESS \
    --function-name reset_counter \
    --from alice \
    --extra-code-path /tmp/counter_app.py \
    --output json -y  \
    --gas 10000000 | dysond query wait-tx --output json | python ../scripts/parse_exec_script_tx.py
out = '\n'.join(out)
result = json.loads(out)
assert result['code'] == 0, f"Error: {result['raw_log']}"

print(f"Counter dApp demo completed successfully!")

## Summary

The Dyson Protocol Storage module provides a powerful and flexible way to store persistent data on-chain. Key takeaways:

1. **Simple API**: Easy-to-use commands for setting, getting, and deleting key-value pairs
2. **Structured Data**: Support for JSON and hierarchical data organization with prefix-based queries
3. **Efficient Retrieval**: Pagination support for handling large datasets
4. **Versatility**: Can store text, JSON, and even binary data (with encoding)
5. **Access Control**: Owner-based write permissions with public read access

These features make the Storage module ideal for a wide range of applications, from simple key-value stores to complex data structures for dApps.